Clonar o repositorio

In [1]:
!git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

fatal: destination path 'carbon-footprint-analysis' already exists and is not an empty directory.


Instalar a versão especifica para o projeto (VsCode x Colab)

In [2]:
pip install scikit-learn==1.8.0

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Importar as bibliotecas

In [3]:
import joblib
import urllib.request
import pandas as pd

Importando o arquivo joblib

In [4]:
import os

# Ver onde você está agora
print("Diretório atual:", os.getcwd())

# Procurar o arquivo .joblib em todo o projeto
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.joblib'):
            print("Encontrado:", os.path.join(root, file))

Diretório atual: c:\Repositorio\carbon-footprint-analysis\notebooks


In [5]:
model = joblib.load(r"../models/best_carbon_footprint_model.joblib")

Analisando o modelo para sanity

In [6]:
print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['consumo_kwh', 'mes']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['estado', 'setor',
                                                   'fonte_energia',
                                                   'season'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=10, n_jobs=-1,
                                       random_state=42))])


In [7]:
encoder = model.named_steps['preprocessor'].named_transformers_['cat']

encoder.categories_

[array(['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG',
        'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR',
        'RS', 'SC', 'SE', 'SP', 'TO'], dtype=object),
 array(['comercial', 'industrial', 'outros', 'residencial', 'rural'],
       dtype=object),
 array(['eólica', 'hidrelétrica', 'nuclear', 'solar', 'térmica'],
       dtype=object),
 array(['Inverno', 'Outono', 'Primavera', 'Verao'], dtype=object)]

Testando o modelo

In [8]:
df = pd.DataFrame({
    "consumo_kwh": [1200],
    "mes": [6],
    "estado": ["SP"],
    "setor": ["industrial"],
    "fonte_energia": ["hydro"],
    "season": ["Inverno"]
})

model.predict(df)

array([12.83992312])

Fazendo o Wrapper


Utilizando o modelo para fazer o comparativo entre todas as fontes de energia eletrica

In [9]:
def predict_all_sources(model, energy_kwh, month, state, usage_type, season):
    sources = ['hidrelétrica', 'nuclear', 'solar', 'térmica', 'eólica']

    results = {}

    for source in sources:

        df = pd.DataFrame({
            "consumo_kwh": [energy_kwh],
            "mes": [month],
            "estado": [state],
            "setor": [usage_type],
            "fonte_energia": [source],
            "season": [season]
        })

        co2 = model.predict(df)[0]

        results[source] = round(float(co2), 2)

    results_sorted = dict(sorted(results.items(), key=lambda x: x[1]))

    return results_sorted

In [10]:
predict_all_sources(
    model,
    energy_kwh=1200,
    month=6,
    state="SP",
    usage_type="industrial",
    season="Inverno"
)

{'hidrelétrica': 4.85,
 'nuclear': 12.84,
 'eólica': 12.84,
 'solar': 54.14,
 'térmica': 987.06}

Criando comparativo para combustiveis liquidos foi utilizado a media de motores motores modernos. E os valores foram retirados do `PCC Guidelines for National Greenhouse Gas Inventories`, `IEA – International Energy Agency` e `US EPA emission factors`

In [11]:
def liquid_fuel_emissions(energy_kwh):

    fuels = {
        "ethanol": {
            "efficiency": 0.27,
            "emission_factor": 0.20
        },
        "gasoline": {
            "efficiency": 0.30,
            "emission_factor": 0.64
        },
        "diesel": {
            "efficiency": 0.38,
            "emission_factor": 0.73
        }
    }

    results = {}

    for fuel, data in fuels.items():

        efficiency = data["efficiency"]
        factor = data["emission_factor"]

        energy_input = energy_kwh / efficiency
        co2 = energy_input * factor

        results[fuel] = round(co2, 2)

    results_sorted = dict(sorted(results.items(), key=lambda x: x[1]))

    return results_sorted

In [12]:
liquid_fuel_emissions(1200)

{'ethanol': 888.89, 'diesel': 2305.26, 'gasoline': 2560.0}

Unificando fontes geradoras e ordenando do menos poluente para o mais poluente adicionado a % de emissão de Co2 que foi aumentado ou diminuido se comparado com a fonte geradora base do brasil `Hydro`

In [13]:
def compare_energy_sources(model, energy_kwh, month, state, usage_type, season):

    electricity = predict_all_sources(
        model,
        energy_kwh,
        month,
        state,
        usage_type,
        season
    )

    fuels = liquid_fuel_emissions(energy_kwh)

    combined = {**electricity, **fuels}

    ranking = dict(sorted(combined.items(), key=lambda x: x[1]))

    base = ranking["hidrelétrica"]

    result = {}

    for source, value in ranking.items():

        percent = ((value - base) / base) * 100

        result[source] = {
            "co2": round(value, 2),
            "vs_hydro_%": round(percent, 2)
        }

    return result

In [14]:
compare_energy_sources(
    model,
    energy_kwh=1200,
    month=6,
    state="SP",
    usage_type="industrial",
    season="Inverno"
)

{'hidrelétrica': {'co2': 4.85, 'vs_hydro_%': 0.0},
 'nuclear': {'co2': 12.84, 'vs_hydro_%': 164.74},
 'eólica': {'co2': 12.84, 'vs_hydro_%': 164.74},
 'solar': {'co2': 54.14, 'vs_hydro_%': 1016.29},
 'ethanol': {'co2': 888.89, 'vs_hydro_%': 18227.63},
 'térmica': {'co2': 987.06, 'vs_hydro_%': 20251.75},
 'diesel': {'co2': 2305.26, 'vs_hydro_%': 47431.13},
 'gasoline': {'co2': 2560.0, 'vs_hydro_%': 52683.51}}

CheckPonit Wrapper

In [15]:
!pip install scikit-learn==1.8.0

!git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

import sys
import os

# Funciona tanto no Colab quanto localmente
colab_path = "/content/carbon-footprint-analysis"
local_path = os.path.join(os.path.dirname(os.getcwd()), '')

if os.path.exists(colab_path):
    sys.path.append(colab_path)
else:
    sys.path.append(local_path)

Defaulting to user installation because normal site-packages is not writeable


fatal: destination path 'carbon-footprint-analysis' already exists and is not an empty directory.


Criando .py do wrapper

Versão Final wrapper

importando o wrapper diretamente do .py para possivel implementação no FastAPI

In [16]:
!pip install scikit-learn==1.8.0

!git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

import sys
import os

# Funciona tanto no Colab quanto localmente
colab_path = "/content/carbon-footprint-analysis"
local_path = os.path.join(os.path.dirname(os.getcwd()), '')

if os.path.exists(colab_path):
    sys.path.append(colab_path)
else:
    sys.path.append(local_path)

Defaulting to user installation because normal site-packages is not writeable


fatal: destination path 'carbon-footprint-analysis' already exists and is not an empty directory.


In [17]:
wrapper_code = '''import joblib
import pandas as pd
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
model = joblib.load(os.path.join(BASE_DIR, "models", "best_carbon_footprint_model.joblib"))


def predict_all_sources(model, energy_kwh, month, state, usage_type, season):
    sources = ['hidrelétrica', 'nuclear', 'solar', 'térmica', 'eólica']
    results = {}

    for source in sources:
        df = pd.DataFrame({
            "consumo_kwh": [energy_kwh],
            "mes": [month],
            "estado": [state],
            "setor": [usage_type],
            "fonte_energia": [source],
            "season": [season]
        })
        co2 = model.predict(df)[0]
        results[source] = round(float(co2), 2)

    return dict(sorted(results.items(), key=lambda x: x[1]))


def liquid_fuel_emissions(energy_kwh):
    fuels = {
        "ethanol":  {"efficiency": 0.27, "emission_factor": 0.20},
        "gasoline": {"efficiency": 0.30, "emission_factor": 0.64},
        "diesel":   {"efficiency": 0.38, "emission_factor": 0.73}
    }
    results = {}
    for fuel, data in fuels.items():
        energy_input = energy_kwh / data["efficiency"]
        co2 = energy_input * data["emission_factor"]
        results[fuel] = round(co2, 2)

    return dict(sorted(results.items(), key=lambda x: x[1]))


def compare_energy_sources(energy_kwh, month, state, usage_type, season):
    electricity = predict_all_sources(model, energy_kwh, month, state, usage_type, season)
    fuels = liquid_fuel_emissions(energy_kwh)
    combined = {**electricity, **fuels}
    ranking = dict(sorted(combined.items(), key=lambda x: x[1]))
    base = ranking["hidrelétrica"]
    result = {}
    for source, value in ranking.items():
        percent = ((value - base) / base) * 100
        result[source] = {"co2": round(value, 2), "vs_hydro_%": round(percent, 2)}
    return result
'''

with open("wrapper.py", "w", encoding="utf-8") as f:
    f.write(wrapper_code)

print("wrapper.py salvo!")

wrapper.py salvo!


In [18]:
import importlib
import wrapper

importlib.reload(wrapper)

<module 'wrapper' from 'c:\\Repositorio\\carbon-footprint-analysis\\notebooks\\wrapper.py'>

Diversos testes para verificar confiabilidade do wrapper

In [19]:
print(wrapper.compare_energy_sources(1200, 6,  "SP", "industrial",  "Inverno"))
print(wrapper.compare_energy_sources(800,  3,  "RJ", "comercial",   "Outono"))     # commercial → comercial
print(wrapper.compare_energy_sources(1500, 12, "MG", "residencial", "Verao"))      # Verão → Verao
print(wrapper.compare_energy_sources(600,  9,  "RS", "industrial",  "Primavera"))
print(wrapper.compare_energy_sources(2000, 1,  "BA", "comercial",   "Verao"))      # commercial → comercial, Verão → Verao
print(wrapper.compare_energy_sources(400,  7,  "SC", "residencial", "Inverno"))
print(wrapper.compare_energy_sources(1000, 5,  "GO", "rural",       "Outono"))
print(wrapper.compare_energy_sources(1800, 10, "PA", "industrial",  "Primavera"))
print(wrapper.compare_energy_sources(500,  4,  "CE", "comercial",   "Outono"))     # commercial → comercial
print(wrapper.compare_energy_sources(2200, 8,  "PR", "industrial",  "Inverno"))

{'hidrelétrica': {'co2': 4.85, 'vs_hydro_%': 0.0}, 'nuclear': {'co2': 12.84, 'vs_hydro_%': 164.74}, 'eólica': {'co2': 12.84, 'vs_hydro_%': 164.74}, 'solar': {'co2': 54.14, 'vs_hydro_%': 1016.29}, 'ethanol': {'co2': 888.89, 'vs_hydro_%': 18227.63}, 'térmica': {'co2': 987.06, 'vs_hydro_%': 20251.75}, 'diesel': {'co2': 2305.26, 'vs_hydro_%': 47431.13}, 'gasoline': {'co2': 2560.0, 'vs_hydro_%': 52683.51}}
{'hidrelétrica': {'co2': 3.33, 'vs_hydro_%': 0.0}, 'nuclear': {'co2': 9.2, 'vs_hydro_%': 176.28}, 'eólica': {'co2': 9.2, 'vs_hydro_%': 176.28}, 'solar': {'co2': 36.35, 'vs_hydro_%': 991.59}, 'ethanol': {'co2': 592.59, 'vs_hydro_%': 17695.5}, 'térmica': {'co2': 649.16, 'vs_hydro_%': 19394.29}, 'diesel': {'co2': 1536.84, 'vs_hydro_%': 46051.35}, 'gasoline': {'co2': 1706.67, 'vs_hydro_%': 51151.35}}
{'hidrelétrica': {'co2': 5.94, 'vs_hydro_%': 0.0}, 'nuclear': {'co2': 16.35, 'vs_hydro_%': 175.25}, 'eólica': {'co2': 16.35, 'vs_hydro_%': 175.25}, 'solar': {'co2': 67.48, 'vs_hydro_%': 1036.03},

In [20]:
# Ver a importância de cada feature no modelo
import pandas as pd
import matplotlib.pyplot as plt

feature_names = (
    list(model.named_steps['preprocessor']
         .named_transformers_['num']
         .get_feature_names_out(['consumo_kwh', 'mes']))
    +
    list(model.named_steps['preprocessor']
         .named_transformers_['cat']
         .get_feature_names_out(['estado', 'setor', 'fonte_energia', 'season']))
)

importances = model.named_steps['regressor'].feature_importances_

fi = pd.DataFrame({'feature': feature_names, 'importance': importances})
fi = fi.sort_values('importance', ascending=False)

print(fi.to_string(index=False))

                   feature   importance
               consumo_kwh 7.581959e-01
     fonte_energia_térmica 2.399369e-01
       fonte_energia_solar 6.637188e-04
                       mes 3.894473e-04
                 estado_SP 8.443492e-05
          season_Primavera 7.420930e-05
                 estado_MG 6.362263e-05
                 estado_RJ 5.450110e-05
             season_Outono 5.042301e-05
            season_Inverno 5.030113e-05
                 estado_BA 4.771467e-05
                 estado_PA 3.893273e-05
fonte_energia_hidrelétrica 3.818308e-05
              season_Verao 3.496292e-05
                 estado_RS 3.440524e-05
                 estado_SC 3.282281e-05
                 estado_PR 2.789202e-05
                 estado_MT 2.702307e-05
                 estado_AL 2.306527e-05
                 estado_ES 2.057830e-05
                 estado_AM 1.675255e-05
                 estado_CE 1.552395e-05
                 estado_GO 1.442649e-05
                 estado_MA 1.318387e-05


In [21]:
import pandas as pd

df = pd.read_csv('../data/processed/synthetic_energy_emissions_dataset.csv')

print(df['fonte_energia'].value_counts())
print()
print(df.groupby('fonte_energia')['emissao_co2'].mean().sort_values())

fonte_energia
hidrelétrica    51723
térmica         22877
eólica          15910
solar            8553
nuclear           937
Name: count, dtype: int64

fonte_energia
hidrelétrica       72.078437
nuclear           198.300270
eólica            203.864340
solar             809.000712
térmica         14729.290801
Name: emissao_co2, dtype: float64


In [22]:
# Ver o CO₂ médio normalizado por consumo
df['co2_por_kwh'] = df['emissao_co2'] / df['consumo_kwh']
print(df.groupby('fonte_energia')['co2_por_kwh'].mean().sort_values())

fonte_energia
hidrelétrica    0.004001
eólica          0.010991
nuclear         0.012008
solar           0.044977
térmica         0.819632
Name: co2_por_kwh, dtype: float64


In [23]:
import pandas as pd
emission_df = pd.read_csv('../data/processed/v2_energy_source_emission_factors.csv')
print(emission_df)

  energy_source  emission_factor
0  hidrelétrica            0.004
1        eólica            0.011
2       nuclear            0.012
3         solar            0.045
4       térmica            0.820
